In [ ]:
from teralizer.config import db_config
import matplotlib as mpl

conn = db_config.get_dev_engine()

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Linux Libertine', 'Libertine', 'Linux Libertine O', 'Times New Roman', 'Times'],
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.linewidth': 1.2,
    'grid.color': '#cccccc',
    'grid.linestyle': '--',
    'grid.linewidth': 0.7,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'savefig.dpi': 300,
    'savefig.format': 'pdf',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

## Projects that used test generalization

In [ ]:
import pandas as pd
from natsort import natsort_keygen

query = """
SELECT id, project_name(id) AS project_name, runtime
FROM project AS p
JOIN v_projects_successes sp ON p.id = sp.project_id
WHERE p.use_test_generalization = true
"""

df = pd.read_sql_query(query, conn)
natsort_key = natsort_keygen()
df = df.sort_values('project_name', key=lambda x: x.map(natsort_key)).reset_index(drop=True)

df

In [ ]:
from IPython.core.display import Markdown
from teralizer.exports import get_dataset_macro, get_table_group_order, get_project_within_type_order, save_latex_table
from natsort import natsorted

# Get project ordering functions
project_within_type_order = get_project_within_type_order()

def seconds_to_hours_minutes_seconds(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours}h {minutes:02d}min {secs:02d}s"

# Sort by proper table order
df_sorted = df.reindex(
    index=natsorted(
        df.index,
        key=lambda x: (
            get_table_group_order(df.loc[x, 'project_name'], 'INITIAL'),
            project_within_type_order.get(df.loc[x, 'project_name'], 99)
        )
    )
)

# Build LaTeX table
latex_table = r"""\begin{table}[H]
  \caption{Total runtimes of Teralizer for all evaluated projects.}
  \label{tab:teralizer-runtimes}
  \begin{tabular}{lr}
    \toprule
    Project & Runtime \\
    \midrule
"""

prev_group_order = None
for _, row in df_sorted.iterrows():
    current_group_order = get_table_group_order(row['project_name'], 'INITIAL')
    
    # Add midrule between different table groups
    if prev_group_order is not None and current_group_order != prev_group_order:
        latex_table += "    \\midrule\n"
    prev_group_order = current_group_order
    
    # Use LaTeX macro for project name
    project_macro = get_dataset_macro(row['project_name'])
    runtime_formatted = seconds_to_hours_minutes_seconds(row['runtime'])
    
    latex_table += f"    {project_macro} & {runtime_formatted} \\\\\n"

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)

# Save to file
save_latex_table(latex_table, 'tab-teralizer-runtimes')

# Also display as DataFrame for reference
df_display = df_sorted[['project_name', 'runtime']].copy()
df_display['runtime'] = df_display['runtime'].apply(seconds_to_hours_minutes_seconds)
df_display = df_display.rename(columns={'project_name': 'Project', 'runtime': 'Runtime'})
display(df_display)

## Runtime requirements for test generalization per project

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from natsort import natsorted

plt.rcParams.update({
    'font.size': 18,           # Default font size for all text
    'axes.titlesize': 20,      # Title font size
    'axes.labelsize': 18,      # Axis label font size
    'xtick.labelsize': 16,     # X tick label font size
    'ytick.labelsize': 16,     # Y tick label font size
    'legend.fontsize': 16,     # Legend font size
})

# Use optimized materialized view instead of complex query with JOINs
query = """
SELECT *
FROM mv_teralizer_runtime_by_stage
ORDER BY project_id, variant_order, stage_group
"""

df_agg = pd.read_sql_query(query, conn)

# Extract base project names using regex (like the original code)
def get_base_project_name(project_name):
    match = re.match(r'^(.*?)(?=-es-)', project_name)
    if match:
        return match.group(1)
    else:
        return project_name

df_agg['base_project_name'] = df_agg['project_name'].apply(get_base_project_name)

# Use categorical data types for better performance and memory usage
ordered_groups = ['Original Validation', 'Specification Extraction', 'Initial Validation', 
                 'Test Transformation', 'Generalization Validation']

df_agg['stage_group'] = pd.Categorical(df_agg['stage_group'], 
                                      categories=ordered_groups, ordered=True)
df_agg['variant'] = pd.Categorical(df_agg['variant'])

# Get unique variants across all data, ordered by variant_order (vectorized)
variant_order_map = df_agg.drop_duplicates('variant').set_index('variant')['variant_order'].to_dict()
all_variants = sorted(df_agg['variant'].unique(), key=lambda v: variant_order_map.get(v, float('inf')))
non_shared_variants = [v for v in all_variants if v != 'SHARED']

# Calculate total runtime per base project for sorting (vectorized)
base_project_totals = df_agg.groupby('base_project_name')['total_runtime'].sum().sort_values(ascending=False)
top_base_projects = base_project_totals.head(10).index.tolist()

# Filter for top 10 base projects using vectorized operations
top_projects_data = df_agg[df_agg['base_project_name'].isin(top_base_projects)].copy()

# Create distinct color mapping
distinct_colors = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', 
    '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', 
    '#98df8a', '#ff9896', '#c5b0d5'
]

if len(all_variants) > len(distinct_colors):
    additional_colors = plt.cm.Set3(np.linspace(0, 1, len(all_variants) - len(distinct_colors)))
    additional_colors = [tuple(c) for c in additional_colors]
    distinct_colors.extend(additional_colors)

color_map = {variant: distinct_colors[i] for i, variant in enumerate(all_variants)}

# Create legend handles
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color_map[variant]) for variant in all_variants]
legend_labels = all_variants

# Define which variants apply to which stage groups, ordered by variant_order
stage_group_variants = {
    'Original Validation': ['SHARED'],
    'Specification Extraction': ['SHARED'],
    'Initial Validation': ['SHARED'],
    'Test Transformation': sorted(non_shared_variants, key=lambda v: variant_order_map.get(v, float('inf'))),
    'Generalization Validation': sorted(non_shared_variants, key=lambda v: variant_order_map.get(v, float('inf')))
}

# Get unique project IDs for plotting, naturally sorted by project_name (vectorized)
project_id_to_name = top_projects_data.drop_duplicates('project_id').set_index('project_id')['project_name'].to_dict()
unique_project_ids = list(top_projects_data['project_id'].unique())
unique_project_ids = natsorted(unique_project_ids, key=lambda pid: project_id_to_name[pid])

# Define parameters for bar positioning
bar_width = 0.3
bar_spacing = 0.05
group_spacing = 0.3

# Count the number of bars in each group
bars_per_group = {group: len(variants) for group, variants in stage_group_variants.items()}

# Calculate the total width of each group (including internal bar spacing)
group_widths = {
    group: (count * bar_width) + ((count - 1) * bar_spacing) if count > 0 else 0
    for group, count in bars_per_group.items()
}

# Calculate the center position of each group
group_centers = {}
current_position = 0
for group in ordered_groups:
    width = group_widths[group]
    group_centers[group] = current_position + width / 2
    current_position += width + group_spacing

# Calculate the position of each bar within its group
bar_positions = {}
for group in ordered_groups:
    variants = stage_group_variants[group]
    num_bars = len(variants)
    if num_bars == 0:
        continue
    group_center = group_centers[group]
    group_width = group_widths[group]
    start_pos = group_center - group_width / 2
    for i, variant in enumerate(variants):
        bar_positions[(group, variant)] = start_pos + i * (bar_width + bar_spacing) + bar_width / 2

# Create pivot tables for all projects to find the maximum bar height
all_pivot_tables = {}
for project_id in unique_project_ids:
    project_data = top_projects_data[top_projects_data['project_id'] == project_id]
    pivot_data = pd.pivot_table(
        project_data,
        index='stage_group',
        columns='variant',
        values='total_runtime',
        aggfunc='sum',
        fill_value=0,
        observed=False
    )
    all_pivot_tables[project_id] = pivot_data

# Find the maximum bar height for each base project
base_project_max_values = {}
for base_name in top_projects_data['base_project_name'].unique():
    max_value = 0
    for project_id in unique_project_ids:
        project_data = top_projects_data[top_projects_data['project_id'] == project_id]
        if project_data.empty:
            continue
        if project_data['base_project_name'].iloc[0] == base_name:
            pivot_data = all_pivot_tables[project_id]
            if not pivot_data.empty:
                project_max = pivot_data.max().max()
                max_value = max(max_value, project_max)
    base_project_max_values[base_name] = max_value * 1.2

# Helper function to format runtime values
def format_runtime(seconds):
    if seconds < 10:
        return f"{seconds:.1f}"
    elif seconds < 100:
        return f"{seconds:.0f}"
    elif seconds < 1000:
        return f"{seconds:.0f}"
    else:
        return f"{seconds/1000:.1f}k"

# Prepare multi-line x-tick labels
xtick_labels = [
    'Original\nValidation' if g == 'Original Validation' else
    'Specification\nExtraction' if g == 'Specification Extraction' else
    'Initial\nValidation' if g == 'Initial Validation' else
    'Test Transformation\n(BASELINE, NAIVE$_{10|50|200}$, IMPROVED$_{10|50|200}$)' if g == 'Test Transformation' else
    'Generalization Validation\n(BASELINE, NAIVE$_{10|50|200}$, IMPROVED$_{10|50|200}$)' if g == 'Generalization Validation' else
    g
    for g in ordered_groups
]

# Plot 1: Runtime by Stage Group for each Project
fig1 = plt.figure(figsize=(17, 3.1*len(unique_project_ids)))
plt.subplots_adjust(hspace=0.3)

for i, project_id in enumerate(unique_project_ids):
    project_data = top_projects_data[top_projects_data['project_id'] == project_id]
    if project_data.empty:
        continue
    project_name = project_data['project_name'].iloc[0]
    base_name = project_data['base_project_name'].iloc[0]
    ax = plt.subplot(len(unique_project_ids), 1, i+1)
    pivot_data = all_pivot_tables[project_id]
    for group in ordered_groups:
        variants = stage_group_variants[group]
        for variant in variants:
            if variant in pivot_data.columns and group in pivot_data.index:
                value = pivot_data.loc[group, variant]
                position = bar_positions[(group, variant)]
                if value > 0:
                    bar = ax.bar(
                        position,
                        value,
                        width=bar_width,
                        color=color_map[variant]
                    )
                    formatted_value = format_runtime(value)
                    ax.text(
                        position,
                        value + (base_project_max_values[base_name] * 0.02),
                        formatted_value,
                        ha='center',
                        va='bottom',
                        fontsize=16,
                        rotation=0
                    )
    ax.set_title(f'Project: {project_name}')
    ax.set_ylabel('Runtime (s)')
    ax.set_xticks([group_centers[group] for group in ordered_groups])
    if i == len(unique_project_ids) - 1:
        ax.set_xticklabels(xtick_labels, rotation=0, ha='center')
    else:
        ax.set_xticklabels([])
    ax.set_xlim(-0.2, current_position - group_spacing + 0.2)
    ax.set_ylim(0, base_project_max_values[base_name])
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# Add horizontal legend at the top
def prettify_variant_label(label):
    return re.sub(r'_(\d+)_TRIES$', r'$_{\1}$', label)

legend_labels = [prettify_variant_label(label) for label in all_variants]

fig1.legend(
    legend_handles,
    legend_labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    ncol=len(legend_labels) # Make legend horizontal
)

plt.tight_layout()
plt.subplots_adjust(top=0.95, right=1)  # Adjust top to fit legend if needed
plt.savefig('fig_teralizer_runtimes.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Export runtime breakdown data as CSV for LLM processing (AGGREGATED DATA)
from teralizer.exports import save_csv_data, standardize_project_name, get_variant_macro, format_runtime_seconds

# Use the aggregated data that's already computed for the figure (df_agg DataFrame)
# This contains meaningful runtime totals per stage group, not raw task records

# Define step order mapping for pipeline execution sequence
step_order_map = {
    'Original Validation': 1,
    'Specification Extraction': 2, 
    'Initial Validation': 3,
    'Test Transformation': 4,
    'Generalization Validation': 5
}

# Prepare runtime breakdown data for CSV export using aggregated data
runtime_breakdown_data = []

for _, row in df_agg.iterrows():
    # Determine if stage is shared (runs once per project) or variant-specific
    is_shared_stage = row['stage_group'] in ['Original Validation', 'Specification Extraction', 'Initial Validation']
    
    runtime_breakdown_data.append({
        'project_name': standardize_project_name(row['project_name']),
        'base_project_name': standardize_project_name(row['base_project_name']),
        'stage_group': row['stage_group'],
        'variant': get_variant_macro(row['variant']),
        'total_runtime_seconds': format_runtime_seconds(row['total_runtime']),
        'is_shared_stage': is_shared_stage,
        'step_order': step_order_map[row['stage_group']]
    })

# Create DataFrame 
runtime_breakdown_df = pd.DataFrame(runtime_breakdown_data)

# Sort for consistent output (by step order, then project, then variant)
runtime_breakdown_df = runtime_breakdown_df.sort_values([
    'step_order', 'base_project_name', 'project_name', 'variant'
]).reset_index(drop=True)

# Save CSV
csv_path = save_csv_data(
    runtime_breakdown_df, 
    'runtime-breakdown-data',
    'Aggregated runtime breakdown showing total time per stage group per project per variant with execution order'
)

print(f"Runtime breakdown data exported to: {csv_path}")
print(f"Shape: {runtime_breakdown_df.shape}")
print(f"Columns: {list(runtime_breakdown_df.columns)}")
print(f"\nSample data:")
display(runtime_breakdown_df.head(10))

print(f"\nPipeline execution order:")
step_summary = runtime_breakdown_df.groupby(['step_order', 'stage_group', 'is_shared_stage']).agg({
    'total_runtime_seconds': ['count', 'mean', 'sum']
}).round(2)
display(step_summary)

print(f"\nSample variant comparison for Test Transformation stage (Step 4):")
test_transform = runtime_breakdown_df[runtime_breakdown_df['stage_group'] == 'Test Transformation'].head(10)
if not test_transform.empty:
    display(test_transform[['project_name', 'variant', 'total_runtime_seconds', 'step_order']].sort_values(['project_name', 'total_runtime_seconds']))

In [ ]:
import pandas as pd

query = f"""
SELECT
    ec.project_name,
    ec.teralizer_variant,
    ec.evosuite_runtime,
    ec.teralizer_runtime,
    ec.evosuite_runtime + ec.teralizer_runtime AS total_runtime,
    ec.evosuite_detected,
    ec.teralizer_detected
FROM mv_efficiency_comparison_evosuite_vs_teralizer ec
WHERE ec.teralizer_variant LIKE '%%_TRIES'
"""
df = pd.read_sql_query(query, conn)
df

In [ ]:
import matplotlib.pyplot as plt
import re

if not df.empty:
    plt.rcParams.update({
        'font.size': 18,           # Default font size for all text
        'axes.titlesize': 20,      # Title font size
        'axes.labelsize': 18,      # Axis label font size
        'xtick.labelsize': 16,     # X tick label font size
        'ytick.labelsize': 16,     # Y tick label font size
        'legend.fontsize': 16,     # Legend font size
    })

    def extract_prefix_and_budget(name):
        match = re.match(r'(.+)-es-default-(\d+s)', name)
        if match:
            return match.group(1), match.group(2)
        return name, None

    # Create a DataFrame with project_prefix and search_budget (do not modify df)
    prefix_budget = df['project_name'].apply(lambda x: pd.Series(extract_prefix_and_budget(x)))
    prefix_budget.columns = ['project_prefix', 'search_budget']
    df_temp = pd.concat([df.reset_index(drop=True), prefix_budget], axis=1)

    project_prefixes = df_temp['project_prefix'].unique()

    # Prepare evosuite_points without modifying df
    evosuite_points = (
        df_temp.groupby(['project_prefix', 'search_budget'])
        .first()
        .reset_index()
    )

    fig, axes = plt.subplots(
        nrows=1, ncols=len(project_prefixes), figsize=(7 * len(project_prefixes), 5)
    )
    if len(project_prefixes) == 1:
        axes = [axes]

    def pareto_front(df, x_col, y_col):
        sorted_df = df.sort_values(x_col)
        pareto = []
        max_y = -float('inf')
        for _, row in sorted_df.iterrows():
            if row[y_col] > max_y:
                pareto.append(row)
                max_y = row[y_col]
        return pd.DataFrame(pareto)

    # Collect mappings for LaTeX tables
    pareto_label_mappings = {project: [] for project in project_prefixes}

    for ax, project in zip(axes, project_prefixes):
        proj_df = df_temp[df_temp['project_prefix'] == project].copy()
        evosuite_df = evosuite_points[evosuite_points['project_prefix'] == project].copy()

        # Plot all points (faded)
        ax.scatter(
            evosuite_df['evosuite_runtime'], evosuite_df['evosuite_detected'],
            marker='o', color='blue', alpha=0.3, s=40, label='EvoSuite Only'
        )
        naive_mask = proj_df['teralizer_variant'].str.startswith('NAIVE')
        ax.scatter(
            proj_df[naive_mask]['total_runtime'], proj_df[naive_mask]['teralizer_detected'],
            marker='x', color='red', alpha=0.3, s=40, label='EvoSuite + NAIVE'
        )
        improved_mask = proj_df['teralizer_variant'].str.startswith('IMPROVED')
        ax.scatter(
            proj_df[improved_mask]['total_runtime'], proj_df[improved_mask]['teralizer_detected'],
            marker='^', color='green', alpha=0.3, s=40, label='EvoSuite + IMPROVED'
        )

        # Prepare all points for Pareto front
        es_points = evosuite_df[['evosuite_runtime', 'evosuite_detected', 'search_budget']].copy()
        es_points['type'] = 'ES'
        es_points['variant'] = None
        es_points.rename(columns={'evosuite_runtime': 'runtime', 'evosuite_detected': 'detected'}, inplace=True)

        naive_points = proj_df[naive_mask][['total_runtime', 'teralizer_detected', 'search_budget', 'teralizer_variant']].copy()
        naive_points['type'] = 'NAIVE'
        naive_points.rename(columns={'total_runtime': 'runtime', 'teralizer_detected': 'detected'}, inplace=True)

        improved_points = proj_df[improved_mask][['total_runtime', 'teralizer_detected', 'search_budget', 'teralizer_variant']].copy()
        improved_points['type'] = 'IMPROVED'
        improved_points.rename(columns={'total_runtime': 'runtime', 'teralizer_detected': 'detected'}, inplace=True)

        all_points = pd.concat([es_points, naive_points, improved_points], ignore_index=True, sort=False)
        pf = pareto_front(all_points, 'runtime', 'detected')

        # Axis formatting
        y_data_min = min(all_points['detected'].min(), pf['detected'].min())
        y_data_max = max(all_points['detected'].max(), pf['detected'].max())
        y_range = y_data_max - y_data_min
        margin = 0.2 * y_range if y_range > 0 else 0.5
        y_min = y_data_min - margin / 2
        y_max = y_data_max + margin
        ax.set_ylim(y_min, y_max)

        ax.set_title("Project: " + project)
        ax.set_xlabel("Runtime (s)")
        ax.set_ylabel("Detected (%)")
        ax.ticklabel_format(style='plain', axis='x')

        # Draw Pareto front line
        ax.plot(
            pf['runtime'], pf['detected'],
            linestyle='--', color='black', linewidth=1.2, zorder=2, label='Pareto front'
        )

        # Plot and label Pareto front points (number only)
        texts = []
        offset = 0.025 * (y_max - y_min)
        for i, (_, row) in enumerate(pf.iterrows(), start=1):
            if row['type'] == 'ES':
                variant_label = r"ES"
                es_budget = row['search_budget']
            elif row['type'] == 'NAIVE':
                variant_label = row['teralizer_variant']
                es_budget = row['search_budget']
            elif row['type'] == 'IMPROVED':
                variant_label = row['teralizer_variant']
                es_budget = row['search_budget']
            else:
                variant_label = "?"
                es_budget = "?"

            detected = row['detected']
            runtime = row['runtime']
            pareto_label_mappings[project].append(
                (i, es_budget, variant_label, detected, runtime)
            )

            # Plotting (unchanged)
            if row['type'] == 'ES':
                color = 'blue'
                marker = 'o'
            elif row['type'] == 'NAIVE':
                color = 'red'
                marker = 'x'
            elif row['type'] == 'IMPROVED':
                color = 'green'
                marker = '^'
            else:
                color = 'black'
                marker = 'o'
            if marker in ['o', '^']:
                ax.scatter(row['runtime'], row['detected'], marker=marker, color=color, s=90, edgecolor='black', zorder=3)
            else:
                ax.scatter(row['runtime'], row['detected'], marker=marker, color=color, s=90, zorder=3)
            texts.append(
                ax.text(
                    row['runtime'], row['detected'] + offset, str(i),
                    fontsize=16, fontweight='bold', color=color, ha='center', va='bottom'
                )
            )

    # Only show one legend, outside the plot area
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig('fig_teralizer_efficiency.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

    # Output LaTeX code for figure + tables
    latex = []
    latex.append(r"\begin{figure}[H]")
    latex.append(r"    \centering")
    latex.append(r"    \includegraphics[width=\textwidth]{fig_teralizer_efficiency.pdf}")
    latex.append(r"    \caption{Pareto fronts for EvoSuite and Teralizer variants across projects.}")
    latex.append(r"    \label{fig:teralizer_efficiency}")

    def latex_variant_label(variant):
        if variant is None or variant == "" or variant == "None" or variant == "ES":
            return "-"
        m_naive = re.match(r"NAIVE_(\d+)_TRIES", str(variant))
        m_improved = re.match(r"IMPROVED_(\d+)_TRIES", str(variant))
        if m_naive:
            return f"NAIVE$_{{{m_naive.group(1)}}}$"
        elif m_improved:
            return f"IMPROVED$_{{{m_improved.group(1)}}}$"
        else:
            return str(variant)

    for idx, project in enumerate(project_prefixes):
        if idx > 0:
            latex.append(r"    \hfill")
        latex.append(r"    \begin{minipage}[t]{0.48\textwidth}")
        latex.append(r"        \centering")
        latex.append(f"        \\captionof{{table}}{{Pareto points for project: {project}.}}")
        latex.append(r"        \begin{tabular}{rrlrr}")
        latex.append(r"            \toprule")
        latex.append(r"            Pt. & EvoSuite & Teralizer & Det. \% & Runtime (s) \\")
        latex.append(r"            \midrule")
        for row in pareto_label_mappings[project]:
            pt, es_budget, variant, detected, runtime = row
            detected_fmt = f"{detected:.1f}"
            runtime_fmt = f"{runtime:.1f}"
            variant_fmt = latex_variant_label(variant)
            latex.append(f"            {pt} & {es_budget} & {variant_fmt} & {detected_fmt} & {runtime_fmt} \\\\")
        latex.append(r"            \bottomrule")
        latex.append(r"        \end{tabular}")
        latex.append(r"    \end{minipage}")
    latex.append(r"\end{figure}")

    print("\n".join(latex))


In [ ]:
# Export Pareto efficiency data as CSV for LLM processing (ALL POINTS - optimal + suboptimal)
from teralizer.exports import save_csv_data, standardize_project_name, format_detection_rate_decimal, format_runtime_seconds

# Extract ALL efficiency data points, not just Pareto optimal ones
if not df.empty:
    # Function to calculate Pareto front (copied from Cell 7)
    def pareto_front(df_points, x_col, y_col):
        sorted_df = df_points.sort_values(x_col)
        pareto = []
        max_y = -float('inf')
        for _, row in sorted_df.iterrows():
            if row[y_col] > max_y:
                pareto.append(row)
                max_y = row[y_col]
        return pd.DataFrame(pareto)
    
    # Prepare helper function to extract project prefix and budget (from Cell 7)
    def extract_prefix_and_budget(name):
        import re
        match = re.match(r'(.+)-es-default-(\d+s)', name)
        if match:
            return match.group(1), match.group(2)
        return name, None
    
    # Process each project separately to get all points
    all_efficiency_data = []
    
    # Get unique project prefixes
    prefix_budget = df['project_name'].apply(lambda x: pd.Series(extract_prefix_and_budget(x)))
    prefix_budget.columns = ['project_prefix', 'search_budget']
    df_temp = pd.concat([df.reset_index(drop=True), prefix_budget], axis=1)
    project_prefixes = df_temp['project_prefix'].unique()
    
    for project_prefix in project_prefixes:
        proj_df = df_temp[df_temp['project_prefix'] == project_prefix].copy()
        evosuite_points = (
            proj_df.groupby(['project_prefix', 'search_budget'])
            .first()
            .reset_index()
        )
        
        # Prepare all points for this project (EvoSuite + Teralizer variants)
        # EvoSuite-only points
        es_points = evosuite_points[['project_prefix', 'search_budget', 'evosuite_runtime', 'evosuite_detected']].copy()
        es_points['type'] = 'ES_ONLY'
        es_points['teralizer_variant'] = 'ES_ONLY'
        es_points.rename(columns={'evosuite_runtime': 'total_runtime', 'evosuite_detected': 'detection_rate'}, inplace=True)
        
        # Teralizer variants (EvoSuite + Teralizer)
        naive_points = proj_df[proj_df['teralizer_variant'].str.startswith('NAIVE', na=False)][
            ['project_prefix', 'search_budget', 'total_runtime', 'teralizer_detected', 'teralizer_variant']
        ].copy()
        naive_points['type'] = 'NAIVE'
        naive_points.rename(columns={'teralizer_detected': 'detection_rate'}, inplace=True)
        
        improved_points = proj_df[proj_df['teralizer_variant'].str.startswith('IMPROVED', na=False)][
            ['project_prefix', 'search_budget', 'total_runtime', 'teralizer_detected', 'teralizer_variant']
        ].copy()
        improved_points['type'] = 'IMPROVED'
        improved_points.rename(columns={'teralizer_detected': 'detection_rate'}, inplace=True)
        
        # Combine all points for this project
        all_points = pd.concat([es_points, naive_points, improved_points], ignore_index=True, sort=False)
        
        # Calculate Pareto front for this project
        pf = pareto_front(all_points, 'total_runtime', 'detection_rate')
        
        # Mark which points are Pareto optimal
        pareto_optimal_indices = set()
        for _, pf_row in pf.iterrows():
            # Find matching points in all_points (need to handle floating point comparison)
            matches = all_points[
                (abs(all_points['total_runtime'] - pf_row['total_runtime']) < 0.1) &
                (abs(all_points['detection_rate'] - pf_row['detection_rate']) < 0.001)
            ]
            pareto_optimal_indices.update(matches.index)
        
        # Add all points to the efficiency data
        for idx, row in all_points.iterrows():
            detection_decimal = format_detection_rate_decimal(row['detection_rate'])
            runtime_secs = format_runtime_seconds(row['total_runtime'])
            detection_per_second = detection_decimal / runtime_secs if runtime_secs > 0 else 0.0
            
            all_efficiency_data.append({
                'project_name': standardize_project_name(row['project_prefix']),
                'evosuite_budget': str(row['search_budget']) if pd.notna(row['search_budget']) else 'unknown',
                'teralizer_variant': row['teralizer_variant'],
                'detection_rate_decimal': detection_decimal,
                'runtime_seconds': runtime_secs,
                'detection_per_second': detection_per_second,
                'is_pareto_optimal': idx in pareto_optimal_indices
            })
    
    # Create DataFrame and save as CSV
    pareto_efficiency_df = pd.DataFrame(all_efficiency_data)
    
    # Sort for consistent output
    pareto_efficiency_df = pareto_efficiency_df.sort_values([
        'project_name', 'evosuite_budget', 'teralizer_variant'
    ]).reset_index(drop=True)
    
    # Save CSV
    csv_path = save_csv_data(
        pareto_efficiency_df, 
        'pareto-efficiency-data',
        'Complete efficiency analysis showing ALL evaluated configurations with Pareto optimality status'
    )
    
    print(f"Pareto efficiency data exported to: {csv_path}")
    print(f"Shape: {pareto_efficiency_df.shape}")
    print(f"Columns: {list(pareto_efficiency_df.columns)}")
    print(f"Sample data:")
    display(pareto_efficiency_df.head(10))
    
    # Show Pareto optimality breakdown
    print(f"\nPareto optimality breakdown:")
    optimality_summary = pareto_efficiency_df.groupby(['project_name', 'is_pareto_optimal']).size().unstack(fill_value=0)
    optimality_summary.columns = ['Suboptimal', 'Pareto Optimal']
    display(optimality_summary)
    
    # Show summary statistics by variant type
    print(f"\nSummary by variant:")
    summary = pareto_efficiency_df.groupby(['teralizer_variant', 'is_pareto_optimal']).agg({
        'detection_rate_decimal': ['mean', 'min', 'max'],
        'runtime_seconds': ['mean', 'min', 'max'],
        'detection_per_second': ['mean', 'min', 'max']
    }).round(3)
    display(summary)
    
else:
    print("No Pareto efficiency data available (df is empty)")

In [ ]:
import pandas as pd

query = f"""
SELECT
    ec.project_name,
    ec.teralizer_variant,
    ec.e_no_validation,
    ec.e_validation,
    ec.t_no_validation,
    ec.t_validation,
    ec.evosuite_detected,
    ec.teralizer_detected
FROM mv_efficiency_comparison_evosuite_vs_teralizer ec
WHERE ec.teralizer_variant LIKE 'IMPROVED_200_TRIES'
"""
df = pd.read_sql_query(query, conn)
df